In [8]:
import pandas as pd

In [7]:
# Dataset available using filepath 'smsspamcollection/sms_spam_collection.txt'
df = pd.read_table('./data/sms_spam_collection.txt',
                   sep='\t',
                   header=None,
                   names=['label_txt', 'sms_message'])

# Output printing out first 5 rows
df.head()

,label_txt,sms_message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
# Encode the text label as a numeric column: ham -> 0, spam -> 1
df['label'] = df['label_txt'].map({'ham': 0, 'spam': 1})
df = df.drop(columns='label_txt')

df.head()

,sms_message,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [3]:
import numpy as np
import ollama

MODEL = 'qwen3-embedding:latest'
DIMENSIONS = 1024
BATCH_SIZE = 100

messages = df['sms_message'].tolist()
vectors = []

for start in range(0, len(messages), BATCH_SIZE):
    batch = ollama.embed(
        model=MODEL,
        input=messages[start:start + BATCH_SIZE],
        dimensions=DIMENSIONS,
    )
    vectors.extend(np.asarray(e, dtype=np.float32) for e in batch.embeddings)
    print(f'\rembedded {len(vectors)}/{len(messages)}', end='')

assert len(vectors) == len(df), f'got {len(vectors)} vectors for {len(df)} rows'

# One float32 vector per row
df['embedding'] = pd.Series(vectors, index=df.index, dtype='object')

df.head()

embedded 5572/5572

,sms_message,label,embedding
0,"Go until jurong point, crazy.. Available only ...",0,"[0.0323555, 0.016009413, 0.06194499, -0.042135..."
1,Ok lar... Joking wif u oni...,0,"[0.08206976, -0.018639985, 0.026879955, -0.093..."
2,Free entry in 2 a wkly comp to win FA Cup fina...,1,"[0.094782084, -0.01949946, 0.01407177, -0.0518..."
3,U dun say so early hor... U c already then say...,0,"[0.07993865, -0.022020226, 0.020707738, -0.054..."
4,"Nah I don't think he goes to usf, he lives aro...",0,"[0.06651712, 0.044909604, 0.010687282, -0.0353..."


In [4]:
# Persist the cleaned dataset, embeddings included, next to the raw one
df.to_parquet('./data/clean_sms_spam_collection.parquet', index=False)
